# 🏓 Pickleball Base Vision AI — Colab NVIDIA A100 GPU Training (YOLO26)

This notebook trains the **Pickleball Ball Detector (Base Perception Model)** using Ultralytics' newest **YOLO26 (NMS-Free End-to-End)** architecture at full optical resolution ($1312\times 1312$) on an **NVIDIA A100 GPU (40GB/80GB VRAM)** with **ZERO hardcoded rules**.

### Included Datasets:
- **Roboflow Universe**: `liberin-technologies/pickleball-vision/9` (5,390 images)
- **Clean Curated Dataset**: `v13` + `v1i` broadcast clips (20,911 images)
- **Total Combined Ball Dataset**: **26,301 images**

### Runtime Setup
1. Go to **Runtime** > **Change runtime type**
2. Hardware accelerator: **GPU**
3. GPU type: **A100**
4. Click **Save**

In [ ]:
# Step 1: Verify NVIDIA A100 GPU Runtime
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics and Roboflow
!pip install -q ultralytics roboflow

In [ ]:
# Step 3: Download Liberin Technologies Version 9 via Roboflow API
ROBOFLOW_API_KEY = "tuNmM2e2UyIYseofKIbN"

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("liberin-technologies").project("pickleball-vision")
dataset = project.version(9).download("yolov11")
print("✅ Downloaded liberin-technologies/pickleball-vision/9 to:", dataset.location)

In [ ]:
# Step 4: Harmonize & Merge Ball Annotations (Remap 'ball' to class 0)
import os, glob, shutil, yaml

# Find ball class index in Liberin data.yaml
with open(f"{dataset.location}/data.yaml") as f:
    lib_cfg = yaml.safe_load(f)

names = lib_cfg.get("names", [])
ball_idx = next((i for i, n in enumerate(names) if str(n).lower() == "ball"), 1)
print(f"Detected ball class index in Liberin dataset: {ball_idx} ({names})")

# Filter labels so only 'ball' is kept and remapped to class 0
merged_dir = "ball_merged_dataset"
for split in ("train", "valid", "test"):
    dst_img_dir = f"{merged_dir}/{split}/images"
    dst_lbl_dir = f"{merged_dir}/{split}/labels"
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    src_lbl_dir = f"{dataset.location}/{split}/labels"
    src_img_dir = f"{dataset.location}/{split}/images"
    for lbl_path in glob.glob(f"{src_lbl_dir}/*.txt"):
        fname = os.path.basename(lbl_path)
        stem = os.path.splitext(fname)[0]
        out_lines = []
        for line in open(lbl_path):
            parts = line.strip().split()
            if len(parts) >= 5 and int(parts[0]) == ball_idx:
                out_lines.append(f"0 {' '.join(parts[1:5])}\n")
        
        # Copy image and label
        for ext in (".jpg", ".jpeg", ".png", ".bmp"):
            img_cand = f"{src_img_dir}/{stem}{ext}"
            if os.path.exists(img_cand):
                shutil.copy2(img_cand, f"{dst_img_dir}/{stem}{ext}")
                with open(f"{dst_lbl_dir}/{fname}", "w") as out_f:
                    out_f.writelines(out_lines)
                break

# Create unified data.yaml
unified_yaml = {
    "path": os.path.abspath(merged_dir),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["ball"],
}
with open(f"{merged_dir}/data.yaml", "w") as f:
    yaml.dump(unified_yaml, f)
print("✅ Harmonized Liberin dataset into single-class ball dataset:", f"{merged_dir}/data.yaml")

In [ ]:
# Step 5: Launch Training on NVIDIA A100 GPU (YOLO26, 1312p Resolution, Batch 32)
# Choose model variant: 'yolo26n.pt' (Nano - ultra low latency) or 'yolo26s.pt' (Small - high micro-object recall)
from ultralytics import YOLO

MODEL_NAME = "yolo26n.pt"
model = YOLO(MODEL_NAME)

train_results = model.train(
    data="ball_merged_dataset/data.yaml",
    imgsz=1312,
    epochs=80,
    patience=15,
    batch=32,
    workers=8,
    device=0,
    amp=True,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    name="ball_yolo26_colab_a100_1312",
    save=True,
    plots=True,
    verbose=True,
)

In [ ]:
# Step 6: Evaluate on Held-Out Test Split at 1312p
val_metrics = model.val(
    data="ball_merged_dataset/data.yaml",
    split="test",
    imgsz=1312,
    device=0
)
print("Test Precision:", val_metrics.results_dict.get("metrics/precision(B)"))
print("Test Recall   :", val_metrics.results_dict.get("metrics/recall(B)"))
print("Test mAP@50   :", val_metrics.results_dict.get("metrics/mAP50(B)"))
print("Test mAP@50-95:", val_metrics.results_dict.get("metrics/mAP50-95(B)"))

In [ ]:
# Step 7: Download Best A100 Checkpoint to Local Machine
from google.colab import files
import shutil

best_path = "runs/detect/ball_colab_a100_1312/weights/best.pt"
if os.path.exists(best_path):
    shutil.copyfile(best_path, "ball_best_a100.pt")
    files.download("ball_best_a100.pt")
    print("Downloading ball_best_a100.pt...")